*Notebook Last Validated: 2026-09-16*

Project UID (Internal, Prod): 864a5789-9a39-48e6-919f-d3260e9d7779

# Hello PyTorch on Rhino FCP

This notebook walks through running the `hello-pt` NVFlare example on Rhino's Federated Computing Platform (FCP), end to end, using the Rhino Python SDK.

**What this example does:** trains a small PyTorch image classifier (on a CIFAR-10-style dataset, 10 classes of small images) across 2 federated rounds. 

This is a "hello world" type example - the goal is to see a real (if tiny) PyTorch training loop work through FCP's federated pipeline, not to produce a production-grade model.

See `README.md` in this directory for the local (non-FCP, Docker-only) walkthrough.

#### Before you run this notebook
1. `pip install rhino_health`
2. Access to a Rhino FCP workgroup and its container registry - see the "Find your workgroup's container registry" section below
3. **Build and push the container image first** (see the next section) - this notebook does not build or push it for you, since that depends on your own registry credentials

## Setup

In [1]:
import json
import os
import zipfile
from getpass import getpass

import rhino_health as rh
from rhino_health.lib.endpoints.code_object.code_object_dataclass import (
    CodeObjectCreateInput,
    CodeTypes,
    ModelTrainInput,
)
from rhino_health.lib.endpoints.dataset.dataset_dataclass import DatasetCreateInput
from rhino_health.lib.endpoints.project.project_dataclass import ProjectCreateInput

# Pin the working directory to this notebook's folder so relative paths resolve
# regardless of where the notebook is opened from (e.g. VS Code, Jupyter Lab, etc.)
os.chdir(os.path.dirname(os.path.abspath(globals().get('__vsc_ipynb_file__', 'notebook.ipynb'))))
print("Setup Complete")

Setup Complete


## Find your workgroup's container registry, then build and push the image

FCP runs your own pre-built image rather than building one from source, so you build and push it *outside* this notebook, using the shared push script in `user-resources/utils/`. You only need to do this once (or again if you change any of the example's code):

1. **Find your workgroup's container repository name** - in the FCP UI, go to **Settings (gear icon) -> Containers & Artifacts -> Workgroup container registry** (there's a copy button next to it).
2. See [Pushing Containers to the ECR](https://docs.rhinofcp.com/getting-started/quick-start-guide/pushing-containers-to-the-ecr) and make sure you've completed all the pre-requisites
3. **Build and push**, from this directory, by entering the following in your command line (replace the elements in <>):
   ```bash
   # cd <PATH_TO>/user-resources/examples/nvflare/hello-pt
   ../../../utils/docker-push.sh <your-workgroup-repo-name> <a-tag-you-choose>
   ```
   Example: `../../../utils/docker-push.sh workgroup-rhino-health-prod DO-335-test-v1`
   
   This can take some time and will print `Done. Container image URI: ...` when it finishes - **copy that exact URI** into `CONTAINER_IMAGE_URI` below (it's what you actually pushed, so it's more reliable than reconstructing the URI yourself).

In [2]:
CONTAINER_IMAGE_URI = "865551847959.dkr.ecr.us-east-1.amazonaws.com/workgroup-rhino-health-prod:DO-335-test-v1" # UPDATE THIS TO THE URI PRINTED BY docker-push.sh

if CONTAINER_IMAGE_URI.startswith("<"):
    raise ValueError("Set CONTAINER_IMAGE_URI to the URI printed by docker-push.sh before continuing.")

print("Completed.")

Completed.


## Authentication

Log in to the Rhino FCP. When prompted, provide your password. This proves who you are to the platform - everything you do below (creating a project, registering data, running training) happens under your account.

**Variables to adjust:**
- `USERNAME` - your Rhino FCP username (typically your email)

In [ ]:
USERNAME = "<YOUR_USERNAME>"  # REPLACE WITH YOUR RHINO FCP USERNAME

print("Logging In")
session = rh.login(username=USERNAME, password=getpass())
user = session.current_user
print("Logged In")

Logging In
Logged In


## Project Creation

Everything on FCP - datasets, code, training runs - lives inside a Project. This creates a new one under your primary workgroup to keep this example's resources separate from anything else you're working on.

Note that every time you rerun this cell, it creates a NEW project (a different `project_uid`), even with the same `PROJECT_NAME`.

In [4]:
PROJECT_NAME = "[User-Example] Hello PyTorch"

project = session.project.add_project(
    ProjectCreateInput(
        name=PROJECT_NAME,
        description="Hello PyTorch NVFlare demo",
        type="Validation",
        primary_workgroup_uid=user.primary_workgroup_uid,
    )
)
print(f"Created project: {project.name}")

Created project: [User-Example] Hello PyTorch


## Dataset Registration

FCP runs one federated learning "client" per registered dataset, so training needs at least one dataset to attach to. This registers a folder of training images - one subfolder per class (`0`-`9`), each full of small images - matching what `app/custom/cifar10trainer.py` expects to find once FCP mounts it.

**Registering a dataset does not upload or extract anything.** `add_dataset(file_base_path=...)` just tells FCP "there's data at this path" - that path is on the filesystem of your **on-prem client** (which may not be the same machine you're running this notebook from), and the files need to already exist there as plain files before you register. Nothing gets zipped/unzipped or copied over the network by this call - the on-prem client's dataset import has no zip-extraction logic at all.

**So, before running the cell below, get the actual image files onto your on-prem client's disk:**
1. Extract `data/train_data.zip` (the same one this notebook extracted locally, above) to get `train/0`, `train/1`, ..., `train/9` folders of `.png` files - the same structure as the "Prepare the training data mount" step in `README.md`'s local walkthrough, just placed on the on-prem client instead of in a local Docker mount.

`unzip <INSERT_PATH>/user-resources/examples/nvflare/hello-pt/data/train_data.zip -d /tmp/hello-pt-extract`
would produce
```
/tmp/hello-pt-extract/
└── train/
    ├── 0/
    │   ├── image0.png
    │   ├── image1.png
    │   └── ... (~10 images)
    ├── 1/
    ├── 2/
    ...
    └── 9/
```

2. Copy that extracted `tmp/hello-pt-extract` folder onto the on-prem client's filesystem (in this case, we're using client-mounted storage), so that `train/0` ... `train/9` sit **directly inside** whatever path you're about to register - i.e. `os.path.join(CLIENT_DATA_PATH, FILE_BASE_PATH)` below needs `train/0`, ..., `train/9` immediately inside it, not the zip file, and not any of `data/`'s other contents (`input_schema.csv`, `output_schema.csv`, `test_data.zip`, `data/train/cohorts/` - none of those are used by this notebook's training flow).

```
/rhino_data/external/import-external-datasets-dev/user-resource-examples/hello-pt/
└── train/
    ├── 0/*.png
    ├── 1/*.png
    ...
    └── 9/*.png
```

**Variables to adjust:**
- `CLIENT_DATA_PATH` - base path on your on-prem client's filesystem where you place data (often a shared "external data" convention specific to your workgroup - check with whoever manages your on-prem client if you're not sure what to use)
- `FILE_BASE_PATH` - subpath under `CLIENT_DATA_PATH` where you've placed *this* example's extracted `train/` folder

In [5]:
CLIENT_DATA_PATH = "/rhino_data/external/import-external-datasets-dev"  # REPLACE WITH YOUR RHINO FCP CLIENT DATA PATH
FILE_BASE_PATH = "user-resource-examples/hello-pt"  # relative to CLIENT_DATA_PATH, one level above the "train" directory
DATASET_NAME = "Hello PyTorch Dataset"

dataset = session.dataset.add_dataset(
    DatasetCreateInput(
        name=DATASET_NAME,
        description="Placeholder CIFAR-10-style training images",
        project_uid=project.uid,
        workgroup_uid=project.primary_workgroup_uid,
        file_base_path=os.path.join(CLIENT_DATA_PATH, FILE_BASE_PATH),
        method="filesystem",
        data_schema=None,
        is_data_deidentified=True,
    )
)
print(f"Registered dataset: {dataset.name} ({dataset.uid})")

Registered dataset: Hello PyTorch Dataset (44b81bab-cb01-4326-a293-090b5c412478)


## NVFlare Code Object Creation

A Code Object is FCP's record of "what code to run." This one points at the image you pushed above, rather than having FCP build one from source the way the auto-container examples do.

In [6]:
code_object_input = CodeObjectCreateInput(
    name="Hello PyTorch",
    description="Hello PyTorch NVFlare demo",
    input_data_schema_uids=[None],
    output_data_schema_uids=[None],
    project_uid=project.uid,
    code_type=CodeTypes.NVIDIA_FLARE_V2_6,  # must match requirements.txt: nvflare==2.6.0 - FCP's provisioning only supports up to v2.6
    config={"container_image_uri": CONTAINER_IMAGE_URI},
)
# return_existing=False + add_version_if_exists=True: if a Code Object with this name already
# exists (e.g. from a prior run), create a NEW VERSION with this config instead of silently
# returning the old one - otherwise a fixed CONTAINER_IMAGE_URI update above would never
# actually take effect on re-runs.
code_object = session.code_object.create_code_object(
    code_object_input, return_existing=False, add_version_if_exists=True
)
code_object = code_object.wait_for_build()  # returns immediately - the image is already built
print(f"Created code object: {code_object.name} ({code_object.uid})")

Created code object: Hello PyTorch (5f949c47-bbbf-44d5-9c0c-1c38037da1b0)


In [8]:
# You can run these cells to inspect the code object and its config, and confirm that the container image URI is correct.
co = session.code_object.get_code_object(code_object.uid)
print(co.config)

{'image_tag': 'DO-335-test-v1', 'image_repo_name_part': 'rhino-health-prod', 'container_image_uri': '865551847959.dkr.ecr.us-east-1.amazonaws.com/workgroup-rhino-health-prod:DO-335-test-v1'}


## NVFlare Federated Training Run

Loads the client/server config files, then kicks off training. There's no encryption or secrets in this example, so `secrets_fed_client`/`secrets_fed_server` are left empty. Waits for the run to complete before proceeding.

In [7]:
with open("app/config/config_fed_client.json") as f:
    config_fed_client = json.dumps(json.load(f))
with open("app/config/config_fed_server.json") as f:
    config_fed_server = json.dumps(json.load(f))

run_params = ModelTrainInput(
    code_object_uid=code_object.uid,
    input_dataset_uids=[dataset.uid],
    one_fl_client_per_dataset=True,
    validation_dataset_uids=[],
    validation_datasets_inference_suffix="",
    timeout_seconds=1200,
    config_fed_client=config_fed_client,
    config_fed_server=config_fed_server,
)

model_train = session.code_object.train_model(run_params)
code_run = model_train.wait_for_completion(1200, poll_frequency=30)
print(f"Training finished with status: {code_run.status}")

Waiting for code run to complete (0 hours 0 minutes and a second)
Waiting for code run to complete (0 hours 0 minutes and 32 seconds)
Waiting for code run to complete (0 hours a minute and 3 seconds)
Waiting for code run to complete (0 hours a minute and 35 seconds)
Waiting for code run to complete (0 hours 2 minutes and 7 seconds)
Waiting for code run to complete (0 hours 2 minutes and 39 seconds)
Waiting for code run to complete (0 hours 3 minutes and 10 seconds)
Done.
Training finished with status: CodeRunStatus.COMPLETED


In [9]:
# If the training failed, you can inspect the errors to see what went wrong.
print(code_run.errors)

[]


## Model Parameters Download

Downloads the trained model weights.

**Note:** You can also download model parameters directly from the FCP UI by going to Code Runs, clicking the 3 dots next to the successful run, and hitting "Download Model Parameters"

In [10]:
weights = session.code_run.get_model_params(code_run.uid)
with open("model_parameters.pt", "wb") as f:
    f.write(weights.getbuffer())
print("Saved model to model_parameters.pt")

Saved model to model_parameters.pt


## Next Steps: Inference (Local Only)

**"Download" vs. "inference" are two different things, happening in two different places:**
- The **Model Parameters Download** cell above just *fetches the trained model file* (`model_parameters.pt`) from FCP onto your own computer via the SDK - a plain file transfer, no computation involved.
- **Inference** means actually *using* that model - feeding it some data and getting predictions/scores back out. That's a separate step, and this notebook doesn't do it on FCP at all: this example only trains on FCP, downloads the result, and inference happens **locally**, by running `infer.py` inside the `hello-pt` Docker container (the same one you built for the "Running this example locally" walkthrough).

(FCP can run inference as its own platform job too, for examples set up to support that - this particular example just doesn't wire that path up, so local Docker is the only way to run inference here.)

**Exactly what to run - you do NOT need step 3 (local training) at all, since you already have a trained model from FCP:**

1. Build the image if you haven't already (`docker build -t hello-pt .`, README step 1).

2. Set up the inference data mount exactly as written in README step 4 - no changes needed there, it's independent of where the model came from:
   ```bash
   mkdir -p ~/hellopt-test/infer_mount/file_data
   cd /tmp
   unzip <path-to-this-repo>/data/test_data.zip -d test_extract
   for d in test_extract/test/*/; do
     cls=$(basename "$d")
     mkdir -p ~/hellopt-test/infer_mount/file_data/$cls
     cp "$d"*.png ~/hellopt-test/infer_mount/file_data/$cls/
   done
   cp /tmp/test_extract/test/cohort_data_test.csv ~/hellopt-test/infer_mount/dataset.csv
   ```

3. **This is the one step that's different from the README:** put the model *this notebook just downloaded* where `/output` will be mounted from, instead of letting local training write one there:
   ```bash
   mkdir -p ~/hellopt-test/output
   cp model_parameters.pt ~/hellopt-test/output/model_parameters.pt
   ```
   (`model_parameters.pt` here is the file sitting in this notebook's own folder, from the download cell above - if you're not sure of its full path, it's wherever this `notebook.ipynb` file lives.)

   **How you know it's using the FCP-trained file, not a locally-trained one:** `infer.py` doesn't guess - you tell it exactly which file to load via the command-line argument (`/output/model_parameters.pt`), and Docker's `-v ~/hellopt-test/output:/output` mount means that path is always whatever's physically sitting in your `~/hellopt-test/output/` folder on your own machine. Since you just copied the FCP-downloaded file there (overwriting anything that was there before, e.g. from a prior local training run), that's what gets used - no ambiguity. If you want to double check, run `ls -la ~/hellopt-test/output/model_parameters.pt` and confirm the timestamp matches when you just copied it.

4. Run inference exactly as in README step 5 - no changes needed:
   ```bash
   docker run -it --rm \
     -v ~/hellopt-test/infer_mount:/input \
     -v ~/hellopt-test/output:/output \
     hello-pt bash
   ```
   Then, inside the container:
   ```bash
   python infer.py /output/model_parameters.pt
   cat /output/dataset.csv   # should show a Model_Score column added
   ```

## Cleanup

**On FCP:** this notebook creates a new Project (containing a Dataset and Code Object) every time you run it. Set `CLEANUP = True` below and re-run that cell to remove them once you're done. (This does not delete the pushed container image from your registry.)

**Locally:** see the "Cleanup" section in `README.md` for removing the local Docker image and test directories.

In [ ]:
CLEANUP = False  # Set to True to delete the Project (and its Dataset/Code Object) created above

if CLEANUP:
    session.code_object.remove_code_object(code_object)
    session.dataset.remove_dataset(dataset)
    session.project.remove_project(project)
    print("Removed code object, dataset, and project from FCP.")
else:
    print("Skipped - set CLEANUP = True above to remove the Project/Dataset/Code Object this notebook created.")

## Additional Resources

- [Rhino SDK Documentation](https://rhinohealth.github.io/rhino_sdk_docs/html/autoapi/index.html)
- [Rhino User Resources](https://github.com/RhinoHealth/user-resources/tree/main)
- [Rhino FCP Platform Documentation](https://docs.rhinofcp.com/)
- [NVFlare GitHub](https://github.com/NVIDIA/NVFlare/tree/main)